Gathering Weather Data from open-meteo

In [0]:
import requests
import pandas as pd

resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 40.6413,
        "longitude": -73.7781,
        "start_date": "2025-07-01",
        "end_date": "2026-06-30",
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall,"
                  "wind_speed_10m,wind_direction_10m,wind_gusts_10m,"
                  "cloud_cover,pressure_msl,weather_code",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "America/New_York",
    },
    timeout=60,
)
resp.raise_for_status()

pdf = pd.DataFrame(resp.json()["hourly"])
pdf["time"] = pd.to_datetime(pdf["time"])

df = spark.createDataFrame(pdf)
df.write.mode("overwrite").saveAsTable("flight_delay_predictor_project.default.jfk_weather_history")
display(df)

Combining Flight and Weather data

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_flights_and_weather_table AS
WITH flights_parsed AS (
  SELECT
    *,
    -- Handle the common BTS date formats: "6/1/2025 12:00:00 AM", "2025-06-01", or an existing timestamp
    COALESCE(
      try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
    ) AS fl_date_ts,
    CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
  FROM flight_delay_predictor_project.default.jfk_flight_data
),
flights_hourly AS (
  SELECT
    *,
    -- Hour bucket, e.g. 1435 -> 14:00 on that date (2400 rolls to the next day)
    timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts) AS dep_hour,
    -- Exact scheduled departure time, handy as a feature later
    timestampadd(MINUTE, crs_dep_int % 100,
                 timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts)) AS crs_dep_ts
  FROM flights_parsed
)
SELECT
  f.*,
  w.* EXCEPT (time)
FROM flights_hourly f
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w
  ON f.dep_hour = w.time;

In [0]:
%sql
SELECT *
FROM flight_delay_predictor_project.default.jfk_flights_and_weather_table

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.first_prediction_table
SELECT weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code
FROM flight_delay_predictor_project.default.jfk_flights_and_weather_table;

In [0]:
%pip install seaborn
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [0]:
df = spark.table('flight_delay_predictor_project.default.first_prediction_table').toPandas()

In [0]:
print(df.head())
print(len(df))

In [0]:
df.info()

In [0]:
from sklearn.model_selection import train_test_split
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
flights_1 = df[df['weather_delay'] != 0]
flights_2 = df[df['weather_delay'] == 0]
print(flights_1.shape)
print(flights_2.shape)

In [0]:
from sklearn.utils import resample

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

In [0]:
X = flights_upsampled.drop('weather_delay',axis=1)
y = flights_upsampled['weather_delay']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)

In [0]:
from sklearn.tree import DecisionTreeClassifier
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
dtree.get_params()

In [0]:
from sklearn.metrics import classification_report,confusion_matrix
predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))

The weather delay column is in minutes and looks very bad for the confusion_matrix and classification report, we will convert the coulmn to hours to make it look better

In [0]:
df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import Window, functions as F

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr"))

In [0]:
hr_delay = hr_delay.toPandas()
print(hr_delay)

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:
flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

In [0]:
X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
from sklearn.metrics import accuracy_score


predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

Testing decision Tree parameters 

In [0]:
classifier = DecisionTreeClassifier(min_samples_split = 20, max_depth = 1000, min_samples_leaf = 2)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy_score(y_test, y_pred)


Random Forest

In [0]:
from sklearn.ensemble import RandomForestClassifier

# This line instantiates the model. 
rf = RandomForestClassifier() 

# Fit the model on your training data.
rf.fit(X_train, y_train) 
y_predict = rf.predict(X_test)

# And score it on your testing data.
accuracy_score(y_test, y_predict)

might be able to delete all cells below

Feature importance

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:
flights_2_downsampled = resample(flights_2,random_state=42,n_samples=845,replace=True)
flights_downsampled = pd.concat([flights_2_downsampled,flights_1])

In [0]:
X = flights_downsampled.drop('delay_in_hr',axis=1)
y = flights_downsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

In [0]:
from sklearn.ensemble import RandomForestClassifier

# This line instantiates the model. 
rf = RandomForestClassifier() 

# Fit the model on your training data.
rf.fit(X_train, y_train) 
y_predict = rf.predict(X_test)

# And score it on your testing data.
accuracy_score(y_test, y_predict)

In [0]:
feature_importances = pd.DataFrame(rf.feature_importances_,
                                   index = X_train.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)
feature_importances